# Tax AI V1.2c — Colab GPU + Gemma 4 E4B + api-ocr-2025

架構：**GitHub Pages → Vercel 免費 HTTPS Proxy → Colab GPU → api-ocr-2025 → Gemma 4 E4B**

V1.2c 修正：
- 固定 `requests==2.32.4`，避免破壞 Google Colab 內建相依。
- Gemma VLM sidecar 改為同一 Notebook process 的 background thread，不再把已載入模型丟到另一個 subprocess。
- 買受人8格=`buyer_tax_id`；右下統一發票專用章=`seller_tax_id`。
- 統編檢查碼只驗證、不反推或修改 OCR/VLM 辨識數字；看不清回 `null`。

api-ocr-2025 pinned commit: `5ef5794c1b0c3fc640d6ac8c8d26562b6c035202`

In [ ]:
import sys, subprocess, torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
subprocess.run(['nvidia-smi'], check=False)
if not torch.cuda.is_available():
    raise RuntimeError('請先在 Colab：執行階段 → 變更執行階段類型 → GPU')

In [ ]:
!apt-get -qq update
!apt-get -qq install -y libzbar0
!pip -q install -U "transformers>=5.5.0" accelerate bitsandbytes huggingface_hub fastapi uvicorn python-multipart pillow "requests==2.32.4"

## 下載 api-ocr-2025
固定使用已驗證的 commit，避免上游更新造成測試結果漂移。

In [ ]:
import os, shutil, subprocess
REPO='/content/api-ocr-2025'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','https://github.com/adi-gov-tw/api-ocr-2025.git',REPO],check=True)
subprocess.run(['git','checkout','5ef5794c1b0c3fc640d6ac8c8d26562b6c035202'],cwd=REPO,check=True)
!pip -q install -r /content/api-ocr-2025/requirements.txt
!pip -q install "requests==2.32.4"

## 載入 Gemma 4 E4B
先嘗試 4-bit 量化以適應免費 Colab GPU。若 Hugging Face 回 401/403，再另外登入 Hugging Face 後重跑本格。

In [ ]:
import gc, torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
MODEL_ID='google/gemma-4-E4B-it'
processor=AutoProcessor.from_pretrained(MODEL_ID)
qconfig=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
try:
    model=AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, quantization_config=qconfig, device_map='auto', dtype=torch.float16
    )
    print('✅ Gemma 4 E4B loaded in 4-bit')
except Exception as e:
    print('⚠️ 4-bit load failed:', repr(e))
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    model=AutoModelForMultimodalLM.from_pretrained(MODEL_ID, device_map='auto', dtype='auto')
    print('✅ Gemma 4 E4B loaded without 4-bit')
print('Model device:', next(model.parameters()).device)

## 建立 OpenAI-compatible Gemma Vision sidecar (:8001)
V1.2c 在**同一個 Colab kernel/process**中啟動 FastAPI，因此直接共用上面已載入的 `model` 與 `processor`。

In [ ]:
import base64, io, re, time, threading, requests
from fastapi import FastAPI, Request
from PIL import Image
import uvicorn

vlm_app=FastAPI(title='Gemma 4 E4B OpenAI-compatible sidecar')

def decode_data_image(value):
    if isinstance(value, dict): value=value.get('url')
    if not isinstance(value, str): return None
    if value.startswith('data:') and ',' in value:
        return Image.open(io.BytesIO(base64.b64decode(value.split(',',1)[1]))).convert('RGB')
    return None

def collect_parts(messages):
    images=[]; texts=[]
    for msg in messages or []:
        content=msg.get('content','')
        if isinstance(content,str):
            texts.append(content)
        elif isinstance(content,list):
            for part in content:
                typ=part.get('type')
                if typ in ('text','input_text'):
                    texts.append(part.get('text',''))
                elif typ in ('image_url','input_image'):
                    value=part.get('image_url') or part.get('image')
                    img=decode_data_image(value)
                    if img is not None: images.append(img)
    return images, '\n'.join(texts)

@vlm_app.get('/health')
async def vlm_health():
    return {'status':'ok','model':MODEL_ID,'cuda':torch.cuda.is_available(),'mode':'same-process-thread'}

@vlm_app.get('/v1/models')
async def vlm_models():
    return {'object':'list','data':[{'id':MODEL_ID,'object':'model'}]}

@vlm_app.post('/v1/chat/completions')
async def vlm_chat(req: Request):
    body=await req.json()
    images,text=collect_parts(body.get('messages',[]))
    guard=(
      'You are a Taiwan unified-invoice vision extraction model. '
      'For triplicate/manual invoices, buyer_tax_id is exactly the eight handwritten/printed boxes immediately after or below 買受人/統一編號 in the upper-left buyer block. '
      'seller_tax_id is the tax ID inside the lower-right 統一發票專用章. Never swap them. '
      'Read all 8 buyer boxes one by one from left to right. Never use checksum to invent, rescue, substitute, or correct digits. '
      'If a digit is genuinely unreadable, return null rather than guessing. Return only the requested JSON/data.'
    )
    if images:
        messages=[{'role':'user','content':[{'type':'image','image':images[0]},{'type':'text','text':guard+'\n'+text}]}]
    else:
        messages=[{'role':'user','content':guard+'\n'+(text or 'Reply OK')}]
    inputs=processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors='pt',
        add_generation_prompt=True, enable_thinking=False
    )
    inputs={k:(v.to(model.device) if hasattr(v,'to') else v) for k,v in inputs.items()}
    input_len=inputs['input_ids'].shape[-1]
    with torch.inference_mode():
        out=model.generate(**inputs,max_new_tokens=min(int(body.get('max_tokens') or 900),1200),do_sample=False)
    raw=processor.decode(out[0][input_len:],skip_special_tokens=True).strip()
    return {
      'id':'chatcmpl-gemma4e4b','object':'chat.completion','created':int(time.time()),'model':MODEL_ID,
      'choices':[{'index':0,'message':{'role':'assistant','content':raw},'finish_reason':'stop'}]
    }

def run_vlm_server():
    config=uvicorn.Config(vlm_app,host='127.0.0.1',port=8001,log_level='warning')
    uvicorn.Server(config).run()

vlm_thread=threading.Thread(target=run_vlm_server,daemon=True)
vlm_thread.start()
for i in range(90):
    try:
        r=requests.get('http://127.0.0.1:8001/health',timeout=2)
        if r.ok:
            print('✅ VLM sidecar:',r.json()); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Gemma sidecar failed to start')

## 啟動 api-ocr-2025 (:8080)

In [ ]:
import os, subprocess, time, requests
env=os.environ.copy()
env.update({
 'APIOCR_VLM_BACKEND':'local',
 'APIOCR_LOCAL_VLM_ENABLED':'true',
 'APIOCR_LOCAL_VLM_URL':'http://127.0.0.1:8001/v1',
 'APIOCR_LOCAL_VLM_MODEL':MODEL_ID,
 'APIOCR_LOCAL_VLM_TIMEOUT':'180',
 'APIOCR_VLM_ENABLED':'true',
 'APIOCR_USE_GPU':'false',
 'APIOCR_DESKEW':'true',
 'APIOCR_UPSCALE_MIN_SIDE':'1400',
 'APIOCR_MIN_CONFIDENCE':'0.20'
})
api_proc=subprocess.Popen(
 ['python','-m','uvicorn','app.main:app','--host','127.0.0.1','--port','8080','--workers','1'],
 cwd=REPO,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True
)
for i in range(120):
    try:
        r=requests.get('http://127.0.0.1:8080/health',timeout=2)
        if r.ok:
            print('✅ api-ocr-2025:',r.json()); break
    except Exception:
        time.sleep(1)
else:
    print('--- api-ocr logs ---')
    try:
        print(api_proc.stdout.read(8000))
    except Exception: pass
    raise RuntimeError('api-ocr-2025 failed to start')

In [ ]:
import requests, json
print('VLM sidecar:', requests.get('http://127.0.0.1:8001/health',timeout=10).json())
print('api-ocr:', requests.get('http://127.0.0.1:8080/health',timeout=10).json())

## 建立 Cloudflare 臨時 HTTPS Tunnel
每次 Colab runtime 重啟，`trycloudflare.com` URL 都會改變。把輸出的 URL 貼回 Tax AI V1.2 前端。

In [ ]:
import os, subprocess, re, time, requests
cf='/content/cloudflared'
if not os.path.exists(cf):
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O',cf],check=True)
    os.chmod(cf,0o755)
tunnel_proc=subprocess.Popen(
 [cf,'tunnel','--url','http://127.0.0.1:8080','--no-autoupdate'],
 stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True
)
public_url=None
deadline=time.time()+90
while time.time()<deadline:
    line=tunnel_proc.stdout.readline()
    if line:
        print(line.rstrip())
        m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',line)
        if m:
            public_url=m.group(0); break
if not public_url:
    raise RuntimeError('Cloudflare Tunnel URL not found')
print('\n===============================================')
print('COLAB_BACKEND_URL =',public_url)
print('===============================================')
print('health =',requests.get(public_url+'/health',timeout=30).json())

## 在 Colab 直接測一張發票（建議先做）
成功後，再把 `COLAB_BACKEND_URL` 貼進網站。

In [ ]:
from google.colab import files
import requests, json
uploaded=files.upload()
for name,data in uploaded.items():
    content_type='image/png' if name.lower().endswith('.png') else 'image/jpeg'
    r=requests.post(
        public_url+'/v1/invoice',
        files={'file':(name,data,content_type)},
        data={'engine':'vlm','slim':'false','include_image':'false'},
        timeout=240
    )
    print('HTTP',r.status_code)
    try: print(json.dumps(r.json(),ensure_ascii=False,indent=2))
    except Exception: print(r.text[:4000])